# Phase 3 — Model Training & Evaluation
## XAI-Based Network Intrusion Detection System

**Author:** Chandra Sekhar Chakraborty  
**Prerequisite:** Run `02_preprocessing.ipynb` first — this notebook reads `data/processed/train_balanced.csv` and `test.csv`  

---

### Training Roadmap
1. [Setup & Load Processed Data](#1-setup--load-processed-data)
2. [Model 1 — Random Forest](#2-model-1--random-forest)
3. [Model 2 — XGBoost](#3-model-2--xgboost)
4. [Model 3 — LSTM](#4-model-3--lstm)
5. [Evaluation — Classification Reports](#5-evaluation--classification-reports)
6. [Confusion Matrices](#6-confusion-matrices)
7. [ROC Curves](#7-roc-curves)
8. [Model Comparison Summary](#8-model-comparison-summary)
9. [Save All Model Artifacts](#9-save-all-model-artifacts)
10. [Phase 3 Handoff to Phase 4](#10-phase-3-handoff-to-phase-4)

## 1. Setup & Load Processed Data

In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

# Sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    accuracy_score, f1_score
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import label_binarize

# XGBoost
from xgboost import XGBClassifier

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# ── Reproducibility ─────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ── Paths ───────────────────────────────────────────
DATA_PROC_DIR = '../data/processed/'
MODELS_DIR    = '../models/'
PLOTS_DIR     = '../docs/model_plots/'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,  exist_ok=True)

# ── Plot style ───────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.edgecolor':   '#0f3460', 'axes.labelcolor': '#e0e0e0',
    'xtick.color':      '#e0e0e0', 'ytick.color':     '#e0e0e0',
    'text.color':       '#e0e0e0', 'grid.color':      '#0f3460',
    'font.family':      'monospace', 'figure.dpi':   120
})
MODEL_COLORS = {'Random Forest': '#00d4ff', 'XGBoost': '#6bcb77', 'LSTM': '#ff6b6b'}

print(f'✅ Libraries loaded')
print(f'   TensorFlow : {tf.__version__}')
print(f'   XGBoost    : {__import__("xgboost").__version__}')

In [ ]:
# ── Load preprocessing metadata ──────────────────────────
meta_path = os.path.join(DATA_PROC_DIR, 'preprocessing_meta.json')
with open(meta_path) as f:
    pre_meta = json.load(f)

FEATURE_COLS = pre_meta['feature_cols']
N_CLASSES    = pre_meta['n_classes']
LABEL_MAP    = pre_meta['label_map']          # {"0": "BENIGN", "1": "DDoS", ...}
CLASS_NAMES  = [LABEL_MAP[str(i)] for i in range(N_CLASSES)]

# Load encoder for decoding predictions
le = joblib.load(os.path.join(MODELS_DIR, 'label_encoder.pkl'))

print('✅ Preprocessing metadata loaded')
print(f'   Features : {len(FEATURE_COLS)}')
print(f'   Classes  : {N_CLASSES}')
print()

# ── Load train / test CSVs ──────────────────────────────
print('Loading processed datasets...')
t0 = time.time()

df_train = pd.read_csv(os.path.join(DATA_PROC_DIR, 'train_balanced.csv'))
df_test  = pd.read_csv(os.path.join(DATA_PROC_DIR, 'test.csv'))

X_train = df_train[FEATURE_COLS].values
y_train = df_train['label'].values.astype(int)
X_test  = df_test[FEATURE_COLS].values
y_test  = df_test['label'].values.astype(int)

print(f'  Train : {X_train.shape[0]:,} rows × {X_train.shape[1]} features  ({time.time()-t0:.1f}s)')
print(f'  Test  : {X_test.shape[0]:,}  rows × {X_test.shape[1]} features')
print(f'  Classes: {N_CLASSES} → {CLASS_NAMES}')

## 2. Model 1 — Random Forest

> **Why Random Forest first?**
> - Fastest to train of the three models
> - Highest macro F1 in preliminary benchmarks
> - SHAP `TreeExplainer` gives exact, millisecond-speed explanations
> - Naturally handles mixed feature ranges (MinMaxScaling still applied for consistency with other models)
>
> **Hyperparameters:** `n_estimators=200`, `class_weight='balanced'` (secondary safety net over SMOTE),
> `n_jobs=-1` (uses all CPU cores).

In [ ]:
print('MODEL 1 — RANDOM FOREST')
print('─' * 55)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=0
)

print('  Training... (this may take 5–15 min on full dataset)')
t0 = time.time()
rf.fit(X_train, y_train)
rf_train_time = time.time() - t0

# Inference
t0 = time.time()
rf_preds = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)
rf_infer_time_ms = (time.time() - t0) / len(X_test) * 1000

print(f'  ✅ Training complete in {rf_train_time:.1f}s')
print(f'  ✅ Inference: {rf_infer_time_ms:.3f} ms/sample')
print(f'  Accuracy : {accuracy_score(y_test, rf_preds)*100:.4f}%')
print(f'  Macro F1 : {f1_score(y_test, rf_preds, average="macro"):.4f}')

## 3. Model 2 — XGBoost

> **Why XGBoost?**
> - Gradient boosting achieves the lowest false positive rate in our benchmarks
> - Handles the CICIDS-2017 feature set extremely well due to built-in regularization
> - SHAP `TreeExplainer` with interaction values for deeper per-alert analysis
>
> We run a **RandomizedSearchCV** (5-fold, 10 iterations) to tune key hyperparameters
> without the full grid search cost. Set `TUNE_XGB = False` to skip tuning and use defaults.

In [ ]:
TUNE_XGB = False   # Set True to run RandomizedSearchCV (adds ~20–40 min)

print('MODEL 2 — XGBOOST')
print('─' * 55)

if TUNE_XGB:
    param_dist = {
        'n_estimators'    : [100, 200, 300],
        'max_depth'       : [4, 6, 8, 10],
        'learning_rate'   : [0.01, 0.05, 0.1],
        'subsample'       : [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'min_child_weight': [1, 3, 5],
        'gamma'           : [0, 0.1, 0.3]
    }
    base_xgb = XGBClassifier(
        use_label_encoder=False, eval_metric='mlogloss',
        tree_method='hist', random_state=RANDOM_SEED, n_jobs=-1
    )
    search = RandomizedSearchCV(
        base_xgb, param_dist, n_iter=10, cv=3,
        scoring='f1_macro', n_jobs=-1,
        random_state=RANDOM_SEED, verbose=1
    )
    search.fit(X_train, y_train)
    xgb = search.best_estimator_
    print(f'  Best params: {search.best_params_}')
else:
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        gamma=0,
        use_label_encoder=False,
        eval_metric='mlogloss',
        tree_method='hist',          # fast histogram-based algorithm
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

print('  Training...')
t0 = time.time()
xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=50)
xgb_train_time = time.time() - t0

t0 = time.time()
xgb_preds = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)
xgb_infer_time_ms = (time.time() - t0) / len(X_test) * 1000

print(f'  ✅ Training complete in {xgb_train_time:.1f}s')
print(f'  ✅ Inference: {xgb_infer_time_ms:.3f} ms/sample')
print(f'  Accuracy : {accuracy_score(y_test, xgb_preds)*100:.4f}%')
print(f'  Macro F1 : {f1_score(y_test, xgb_preds, average="macro"):.4f}')

## 4. Model 3 — LSTM

> **Why LSTM?**
> - Slow-rate temporal attacks (Slowloris, GoldenEye) build over consecutive flows.
>   RF and XGBoost see each flow in isolation — LSTM sees a sequence of 5 consecutive flows.
>
> **Input reshaping:** CICIDS-2017 is tabular, but we simulate temporal context by grouping
> consecutive flows into sequences of `TIME_STEPS=5`.
> Each sample becomes shape `(5, n_features)` → LSTM input is `(batch, 5, n_features)`.
>
> **Callbacks:**
> - `EarlyStopping` — stops if val_loss doesn't improve for 5 epochs (saves training time)
> - `ReduceLROnPlateau` — halves learning rate if val_loss plateaus for 3 epochs
> - `ModelCheckpoint` — saves best weights automatically

In [ ]:
TIME_STEPS = 5

def reshape_for_lstm(X, y, time_steps):
    """Group consecutive rows into (time_steps, features) sequences.
    Truncates the dataset to be divisible by time_steps.
    The label for each sequence is the label of its LAST row.
    """
    n = len(X) - (len(X) % time_steps)  # truncate to divisible length
    X_seq = X[:n].reshape(-1, time_steps, X.shape[1])
    y_seq = y[:n:time_steps]             # one label per sequence
    # Use the label of the LAST step in each window
    y_seq = np.array([y[i + time_steps - 1] for i in range(0, n - time_steps + 1, time_steps)])
    return X_seq, y_seq

print('MODEL 3 — LSTM')
print('─' * 55)
print(f'  TIME_STEPS = {TIME_STEPS}')
print(f'  Reshaping X_train: {X_train.shape} → ...') 

X_train_seq, y_train_seq = reshape_for_lstm(X_train, y_train, TIME_STEPS)
X_test_seq,  y_test_seq  = reshape_for_lstm(X_test,  y_test,  TIME_STEPS)

print(f'  X_train_seq : {X_train_seq.shape}  (samples, time_steps, features)')
print(f'  X_test_seq  : {X_test_seq.shape}')
print(f'  y_train_seq : {y_train_seq.shape}')

# One-hot encode labels for categorical crossentropy
y_train_cat = to_categorical(y_train_seq, num_classes=N_CLASSES)
y_test_cat  = to_categorical(y_test_seq,  num_classes=N_CLASSES)
print(f'  y_train_cat : {y_train_cat.shape}  (one-hot encoded)')

In [ ]:
n_features = X_train_seq.shape[2]

def build_lstm(time_steps, n_features, n_classes):
    model = Sequential([
        LSTM(128, input_shape=(time_steps, n_features),
             return_sequences=True, name='lstm_1'),
        Dropout(0.3, name='dropout_1'),
        BatchNormalization(name='bn_1'),
        LSTM(64, return_sequences=False, name='lstm_2'),
        Dropout(0.3, name='dropout_2'),
        BatchNormalization(name='bn_2'),
        Dense(64, activation='relu', name='dense_1'),
        Dropout(0.2, name='dropout_3'),
        Dense(n_classes, activation='softmax', name='output')
    ], name='LSTM_NIDS')
    return model

lstm_model = build_lstm(TIME_STEPS, n_features, N_CLASSES)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

In [ ]:
LSTM_MODEL_PATH = os.path.join(MODELS_DIR, 'lstm_best.keras')

callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        LSTM_MODEL_PATH, monitor='val_loss',
        save_best_only=True, verbose=0
    )
]

print('  Training LSTM... (this may take 30–60 min on full dataset)')
print('  Tip: set SAMPLE_MODE=True in Phase 2 for fast iteration')
t0 = time.time()

history = lstm_model.fit(
    X_train_seq, y_train_cat,
    epochs=30,
    batch_size=1024,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)
lstm_train_time = time.time() - t0

# Inference
t0 = time.time()
lstm_proba = lstm_model.predict(X_test_seq, batch_size=2048, verbose=0)
lstm_preds_seq = np.argmax(lstm_proba, axis=1)
lstm_infer_time_ms = (time.time() - t0) / len(X_test_seq) * 1000

print(f'  ✅ Training complete in {lstm_train_time:.1f}s')
print(f'  ✅ Inference: {lstm_infer_time_ms:.3f} ms/sample')
print(f'  Accuracy : {accuracy_score(y_test_seq, lstm_preds_seq)*100:.4f}%')
print(f'  Macro F1 : {f1_score(y_test_seq, lstm_preds_seq, average="macro"):.4f}')

In [ ]:
# ── Plot LSTM training history ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=13, fontweight='bold', color='#00d4ff')

for ax, metric, title in zip(axes, ['loss', 'accuracy'], ['Loss', 'Accuracy']):
    ax.plot(history.history[metric],          label=f'Train {title}', color='#ff6b6b', lw=2)
    ax.plot(history.history[f'val_{metric}'], label=f'Val {title}',   color='#00d4ff', lw=2)
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.set_title(f'LSTM {title}', fontsize=11, color='#e0e0e0')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '01_lstm_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → docs/model_plots/01_lstm_training_history.png')

## 5. Evaluation — Classification Reports

> Full per-class metrics for all three models: precision, recall, F1, support.
> We also compute Detection Rate (DR) and False Alarm Rate (FAR) — the metrics that matter most in a SOC.

In [ ]:
def detection_rate(y_true, y_pred):
    """DR = TP / (TP + FN) for attack classes only (exclude BENIGN)."""
    # Assume class 0 = BENIGN (index depends on LabelEncoder ordering)
    benign_id = list(le.classes_).index('BENIGN') if 'BENIGN' in le.classes_ else 0
    attack_mask = y_true != benign_id
    if attack_mask.sum() == 0: return 0.0
    tp = ((y_pred[attack_mask]) != benign_id).sum()
    return tp / attack_mask.sum()

def false_alarm_rate(y_true, y_pred):
    """FAR = FP / (FP + TN) for benign class only."""
    benign_id = list(le.classes_).index('BENIGN') if 'BENIGN' in le.classes_ else 0
    benign_mask = y_true == benign_id
    if benign_mask.sum() == 0: return 0.0
    fp = ((y_pred[benign_mask]) != benign_id).sum()
    return fp / benign_mask.sum()

# ── Results dictionary ────────────────────────────────────
results = {}

models_eval = [
    ('Random Forest', rf_preds,       rf_proba,   y_test,     rf_train_time,   rf_infer_time_ms),
    ('XGBoost',       xgb_preds,      xgb_proba,  y_test,     xgb_train_time,  xgb_infer_time_ms),
    ('LSTM',          lstm_preds_seq, lstm_proba, y_test_seq, lstm_train_time, lstm_infer_time_ms),
]

for name, preds, proba, y_true_m, train_t, infer_t in models_eval:
    # Match class names to classes present in y_true_m
    present_ids = sorted(np.unique(y_true_m))
    present_names = [CLASS_NAMES[i] for i in present_ids]

    acc    = accuracy_score(y_true_m, preds)
    macro_f1 = f1_score(y_true_m, preds, average='macro')
    dr     = detection_rate(y_true_m, preds)
    far    = false_alarm_rate(y_true_m, preds)

    # Macro AUC (one-vs-rest, using proba)
    y_bin = label_binarize(y_true_m, classes=present_ids)
    proba_filtered = proba[:, present_ids] if proba.shape[1] > len(present_ids) else proba
    try:
        macro_auc = roc_auc_score(y_bin, proba_filtered, multi_class='ovr', average='macro')
    except Exception:
        macro_auc = float('nan')

    results[name] = {
        'accuracy'    : acc,
        'macro_f1'    : macro_f1,
        'dr'          : dr,
        'far'         : far,
        'macro_auc'   : macro_auc,
        'train_time_s': train_t,
        'infer_ms'    : infer_t,
        'preds'       : preds,
        'proba'       : proba,
        'y_true'      : y_true_m,
        'present_ids' : present_ids
    }

    print(f'\n{"─" * 60}')
    print(f'  MODEL: {name}')
    print(f'{"─" * 60}')
    print(f'  Accuracy         : {acc*100:.4f}%')
    print(f'  Macro F1         : {macro_f1:.4f}')
    print(f'  Detection Rate   : {dr*100:.4f}%')
    print(f'  False Alarm Rate : {far*100:.4f}%')
    print(f'  Macro AUC (OVR)  : {macro_auc:.4f}')
    print(f'  Train Time       : {train_t:.1f}s')
    print(f'  Inference/sample : {infer_t:.3f} ms')
    print(f'\n  Per-class report:')
    print(classification_report(y_true_m, preds, target_names=present_names, zero_division=0))

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
fig.suptitle('Confusion Matrices — All Three Models', fontsize=14,
             fontweight='bold', color='#00d4ff')

for ax, (name, res) in zip(axes, results.items()):
    present_ids   = res['present_ids']
    present_names = [CLASS_NAMES[i] for i in present_ids]
    cm = confusion_matrix(res['y_true'], res['preds'], labels=present_ids, normalize='true')
    sns.heatmap(
        cm, annot=True, fmt='.2f', cmap='YlOrRd',
        xticklabels=[n[:12] for n in present_names],
        yticklabels=[n[:12] for n in present_names],
        ax=ax, cbar=False, linewidths=0.3, linecolor='#1a1a2e',
        annot_kws={'size': 6}
    )
    ax.set_title(f'{name}\nAcc={res["accuracy"]*100:.2f}%  F1={res["macro_f1"]:.3f}',
                 fontsize=10, color=MODEL_COLORS.get(name, '#e0e0e0'))
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True', fontsize=8)
    ax.tick_params(labelsize=6)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '02_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → docs/model_plots/02_confusion_matrices.png')

## 7. ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle('ROC Curves — All Three Models (One-vs-Rest per Class)',
             fontsize=13, fontweight='bold', color='#00d4ff')

for ax, (name, res) in zip(axes, results.items()):
    present_ids   = res['present_ids']
    present_names = [CLASS_NAMES[i] for i in present_ids]
    y_bin  = label_binarize(res['y_true'], classes=present_ids)
    proba  = res['proba'][:, present_ids] if res['proba'].shape[1] > len(present_ids) else res['proba']
    color_cycle = plt.cm.tab20(np.linspace(0, 1, len(present_ids)))

    micro_fpr, micro_tpr, _ = roc_curve(y_bin.ravel(), proba.ravel())
    micro_auc = auc(micro_fpr, micro_tpr)
    ax.plot(micro_fpr, micro_tpr, lw=2.5, linestyle='--',
            color=MODEL_COLORS.get(name, '#ffffff'),
            label=f'Micro-avg AUC = {micro_auc:.4f}')

    for i, (cls_id, cls_name, color) in enumerate(zip(present_ids, present_names, color_cycle)):
        if y_bin[:, i].sum() == 0: continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=1, color=color, alpha=0.7,
                label=f'{cls_name[:14]} ({roc_auc:.3f})')

    ax.plot([0,1],[0,1],'--', color='#555', lw=1, alpha=0.5)
    ax.set_xlim([0, 0.05])   # zoom to low FPR region (SOC-relevant)
    ax.set_xlabel('False Positive Rate (zoomed 0–5%)', fontsize=8)
    ax.set_ylabel('True Positive Rate', fontsize=8)
    ax.set_title(f'{name}\nMicro-AUC = {micro_auc:.4f}',
                 fontsize=10, color=MODEL_COLORS.get(name, '#e0e0e0'))
    ax.legend(fontsize=5.5, loc='lower right', ncol=2)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '03_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → docs/model_plots/03_roc_curves.png')

## 8. Model Comparison Summary

In [ ]:
print('=' * 70)
print('MODEL COMPARISON — CICIDS-2017 TEST SET')
print('=' * 70)
print(f'  {"Metric":<25} {"Random Forest":>16} {"XGBoost":>16} {"LSTM":>16}')
print('  ' + '─' * 78)

metrics = [
    ('Accuracy (%)',       lambda r: f'{r["accuracy"]*100:.4f}%'),
    ('Macro F1',           lambda r: f'{r["macro_f1"]:.4f}'),
    ('Detection Rate (%)', lambda r: f'{r["dr"]*100:.4f}%'),
    ('False Alarm Rate (%)',lambda r: f'{r["far"]*100:.4f}%'),
    ('Macro AUC (OVR)',    lambda r: f'{r["macro_auc"]:.4f}'),
    ('Train Time (s)',     lambda r: f'{r["train_time_s"]:.1f}s'),
    ('Inference (ms/smpl)',lambda r: f'{r["infer_ms"]:.3f}ms'),
]

for metric_name, fn in metrics:
    rf_val  = fn(results['Random Forest'])
    xgb_val = fn(results['XGBoost'])
    lst_val = fn(results['LSTM'])
    print(f'  {metric_name:<25} {rf_val:>16} {xgb_val:>16} {lst_val:>16}')

print('=' * 70)

In [ ]:
metrics_plot = ['accuracy', 'macro_f1', 'dr', 'macro_auc']
labels_plot  = ['Accuracy', 'Macro F1', 'Detection Rate', 'Macro AUC']
model_names  = list(results.keys())

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Model Comparison — Key Metrics', fontsize=13, fontweight='bold', color='#00d4ff')

for ax, m, label in zip(axes, metrics_plot, labels_plot):
    vals   = [results[n][m] for n in model_names]
    colors = [MODEL_COLORS[n] for n in model_names]
    bars   = ax.bar(model_names, vals, color=colors, edgecolor='none', width=0.5)
    ax.set_title(label, fontsize=11, color='#e0e0e0')
    ax.set_ylim(min(vals) * 0.98, 1.0)
    ax.tick_params(labelsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8, color='#e0e0e0')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '04_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved → docs/model_plots/04_model_comparison.png')

## 9. Save All Model Artifacts

In [ ]:
print('SAVING MODEL ARTIFACTS')
print('─' * 55)

# Random Forest
rf_path = os.path.join(MODELS_DIR, 'random_forest.pkl')
joblib.dump(rf, rf_path, compress=3)
print(f'  ✅ Random Forest  → {rf_path}  ({os.path.getsize(rf_path)/1024**2:.1f} MB)')

# XGBoost
xgb_path = os.path.join(MODELS_DIR, 'xgboost_model.pkl')
joblib.dump(xgb, xgb_path, compress=3)
print(f'  ✅ XGBoost        → {xgb_path}  ({os.path.getsize(xgb_path)/1024**2:.1f} MB)')

# LSTM (already saved by ModelCheckpoint as best weights)
lstm_model.save(os.path.join(MODELS_DIR, 'lstm_model.keras'))
print(f'  ✅ LSTM           → {MODELS_DIR}lstm_model.keras')

# Save results dict (without large arrays) as JSON
results_summary = {
    name: {
        k: float(v) if isinstance(v, float) else v
        for k, v in res.items()
        if k not in ('preds', 'proba', 'y_true', 'present_ids')
    }
    for name, res in results.items()
}
results_path = os.path.join(DATA_PROC_DIR, 'model_results.json')
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f'  ✅ model_results.json → {results_path}')

# Verify all artifacts
print('\n  ARTIFACT VERIFICATION:')
for path in [rf_path, xgb_path,
             os.path.join(MODELS_DIR, 'lstm_model.keras'),
             os.path.join(MODELS_DIR, 'lstm_best.keras'),
             results_path]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/(1024**2) if exists else 0
    print(f'  {"✅" if exists else "❌"}  {os.path.basename(path):35s}  {size:.2f} MB')

## 10. Phase 3 Handoff to Phase 4

In [ ]:
print('=' * 65)
print('PHASE 3 — MODEL TRAINING COMPLETE')
print('=' * 65)

rf_r  = results['Random Forest']
xgb_r = results['XGBoost']
lst_r = results['LSTM']

best_f1_model = max(results, key=lambda n: results[n]['macro_f1'])
best_far_model = min(results, key=lambda n: results[n]['far'])

print(f'''
RESULTS AT A GLANCE:
  Best Macro F1    : {best_f1_model}  ({results[best_f1_model]['macro_f1']:.4f})
  Lowest FAR       : {best_far_model}  ({results[best_far_model]['far']*100:.4f}%)
  Fastest Inference: XGBoost ({xgb_r['infer_ms']:.3f} ms/sample)

ARTIFACTS SAVED:
  models/random_forest.pkl       ← Phase 4 SHAP TreeExplainer input
  models/xgboost_model.pkl       ← Phase 4 SHAP TreeExplainer input
  models/lstm_model.keras        ← Phase 4 SHAP DeepExplainer input
  models/lstm_best.keras         ← Best weights checkpoint
  data/processed/model_results.json

PLOTS SAVED:
  docs/model_plots/01_lstm_training_history.png
  docs/model_plots/02_confusion_matrices.png
  docs/model_plots/03_roc_curves.png
  docs/model_plots/04_model_comparison.png

NEXT: notebooks/04_xai_shap.ipynb
  → SHAP TreeExplainer for RF and XGBoost
  → SHAP DeepExplainer for LSTM
  → Per-sample waterfall charts
  → Global beeswarm summary plots
  → Dependence plots for top feature pairs
''')
print('=' * 65)